In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# A CNN has 3 types of layers:

# ── 1. CONV layer — feature detector ─────────────────────

conv = nn.Conv2d(in_channels=1,   # grayscale = 1 channel
                 out_channels=32, # learn 32 different filters
                 kernel_size=3,   # each filter is 3x3
                 padding=1)       # pad edges so output same size


# Input:  (batch, 1,  28, 28)
# Output: (batch, 32, 28, 28) 

# ── 2. POOLING layer — downsample ────────────────────────
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# ── 3. FULLY CONNECTED — classify ────────────────────────
# After conv layers, flatten and classify
fc = nn.Linear(32 * 7 * 7, 10)
# Input: (batch, 32*7*7) = (batch, 1568)
# Output: (batch, 10)

#  Full CNN Architecture from MNIST

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block 1: conv → relu → pool
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)

        # Block 2: conv → relu → pool
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        # Classifier
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)
        
        
    def forward(self, x):
        # x: (batch, 1, 28, 28)

        x = self.pool(F.relu(self.conv1(x)))
        # → (batch, 32, 14, 14)

        x = self.pool(F.relu(self.conv2(x)))
        # → (batch, 64, 7, 7)

        x = x.view(x.size(0), -1)
        # → (batch, 3136)  flatten for FC layers

        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        # → (batch, 10)
        
        return x
    
    
#  Checking Architecture


model = CNN()
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

# Trace shapes manually
x = torch.randn(1, 1, 28, 28)  # one MNIST image
print(f"\nInput:         {x.shape}")
x = model.pool(F.relu(model.conv1(x)))
print(f"After block 1: {x.shape}")
x = model.pool(F.relu(model.conv2(x)))
print(f"After block 2: {x.shape}")
x = x.view(x.size(0), -1)
print(f"After flatten: {x.shape}")

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (drop): Dropout(p=0.25, inplace=False)
)

Parameters: 421,642

Input:         torch.Size([1, 1, 28, 28])
After block 1: torch.Size([1, 32, 14, 14])
After block 2: torch.Size([1, 64, 7, 7])
After flatten: torch.Size([1, 3136])
